# Agentic RAG Demo
This notebook demonstrates the Adaptive RAG, Corrective RAG (CRAG), and Self-RAG pipelines working together.

In [16]:
import sys
from pathlib import Path

# Ensure project root is in path so 'src' imports work
PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

from main import agentic_rag_app


## 1. Direct LLM Route
For general queries, the Adaptive Router bypasses retrieval and uses the LLM directly.

In [18]:
query = "what is 2+2?"

initial_state = {
    "query": query,
    "original_query": query,
    "namespace": "default_namespace",
    "documents": [],
    "final_context_strips": [],
    "needs_web_search": False,
    "answer": "",
    "retries": 0,
    "route": ""
}

for chunk in agentic_rag_app.stream(initial_state):
    for node, update in chunk.items():
        print(f"\n===> Update from node [{node}]:")
        if 'route' in update:
            print(f"Route chosen: {update['route']}")
        if 'answer' in update and update['answer']:
            print(f"Answer: \n{update['answer']}\n")



===> Update from node [router]:
Route chosen: llm_direct

===> Update from node [generate]:
Answer: 
2 + 2 = 4.



## 2. Vectorstore Route (CRAG + Self-RAG)
For domain specific queries, the Adaptive Router will choose vectorstore. Then CRAG assesses documents and Self-RAG checks hallucinations/quality.

In [19]:
query = "Tell me about the cross-modal architecture of RAG-Anything."

initial_state_2 = {
    "query": query,
    "original_query": query,
    "namespace": "default_namespace",
    "documents": [],
    "final_context_strips": [],
    "needs_web_search": False,
    "answer": "",
    "retries": 0,
    "route": ""
}

for chunk in agentic_rag_app.stream(initial_state_2):
    for node, update in chunk.items():
        print(f"\n===> Update from node [{node}]:")
        if 'documents' in update:
            print(f"Retrieved {len(update['documents'])} documents.")
        if 'final_context_strips' in update:
            print(f"Relevant context strips: {len(update['final_context_strips'])}")
        if 'needs_web_search' in update:
            print(f"Needs web search fallback: {update['needs_web_search']}")
        if 'answer' in update and update['answer']:
            print(f"Answer: \n{update['answer']}\n")



===> Update from node [router]:

===> Update from node [retrieve]:
Retrieved 3 documents.

===> Update from node [evaluate]:
Relevant context strips: 12
Needs web search fallback: False

===> Update from node [generate]:
Answer: 
**Cross‑modal architecture of RAG‑Anything**

RAG‑Anything is built around a **dual‑graph construction strategy** that lets the system treat every document as a *multimodal knowledge graph* and then merge those per‑document graphs into a global, cross‑modal retrieval index. The architecture can be broken down into three logical layers:

| Layer | Main Function | Key Modules |
|-------|----------------|-------------|
| **1️⃣ Parallel Parsing Layer** | Extracts structured information from each modality while preserving the original layout and positional context. | • **Hierarchical Text Extraction** – OCR → layout‑aware text blocks → headings, paragraphs, lists.<br>• **Image Caption & Metadata Extraction** – vision encoder (e.g., CLIP/ViT) + captioning model → c

## 3. Web Search Route (Fallback)
For current events, the adaptive router or fallback logic triggers a web search.

In [20]:
query = "What is the weather like of Delhi?"

initial_state_3 = {
    "query": query,
    "original_query": query,
    "namespace": "default_namespace",
    "documents": [],
    "final_context_strips": [],
    "needs_web_search": False,
    "answer": "",
    "retries": 0,
    "route": ""
}

for chunk in agentic_rag_app.stream(initial_state_3):
    for node, update in chunk.items():
        print(f"\n===> Update from node [{node}]:")
        if 'route' in update:
            print(f"Route chosen: {update['route']}")
        if 'final_context_strips' in update:
            print(f"Context obtained.")
        if 'answer' in update and update['answer']:
            print(f"Answer: \n{update['answer']}\n")



===> Update from node [router]:
Route chosen: web_search

===> Update from node [web_search]:
Context obtained.

===> Update from node [generate]:
Answer: 
**Current weather in Delhi (as reported in the latest update):**

- **Temperature:** 29.2 °C  
- **Feels‑like:** 27.2 °C  
- **Condition:** Mist  
- **Humidity:** 40 %  
- **Wind:** 19.1 km/h (light to moderate breeze)  
- **UV Index:** 0 (very low)  
- **Visibility:** 5 km  
- **Sunrise:** 06:17 AM  
- **Sunset:** 06:36 PM  

So, Delhi is experiencing a mild‑warm day with misty conditions, relatively low humidity, a gentle breeze, and excellent UV protection. Visibility is moderate at about 5 km, and daylight runs from early morning (06:17) to early evening (06:36).

